<a href="https://colab.research.google.com/github/mena-04/DoS-Stress-Testing/blob/main/testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!nvidia-smi
!pip uninstall -y torch torchvision torchaudio vllm
!pip install -q -U uv
!uv pip install --system vllm --torch-backend=cu130

print("vLLM:", vllm.__version__)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Fri Sep 11 13:59:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

In [1]:
import torch
import vllm

print("vLLM:", vllm.__version__)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

vLLM: 0.29.0
Torch: 2.13.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: Tesla T4


In [2]:
import importlib.util

print("torchaudio installed:",
      importlib.util.find_spec("torchaudio") is not None)

torchaudio installed: True


In [3]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [4]:
import subprocess, time, requests, os, signal

# Stop old server if it exists
try:
    server.terminate()
    server.wait(timeout=10)
except:
    pass

log = open("/content/vllm.log", "w")

server = subprocess.Popen(
    [
        "vllm", "serve",
        "Qwen/Qwen2.5-0.5B-Instruct",
        "--dtype", "half",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.85",
        "--max-num-seqs", "8",
        "--host", "127.0.0.1",
        "--port", "8000",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("PID:", server.pid)

PID: 6060


In [5]:
for i in range(90):
    if server.poll() is not None:
        print("SERVER PROCESS EXITED")
        break

    try:
        r = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if r.status_code == 200:
            print("vLLM READY")
            break
    except:
        pass

    time.sleep(2)

vLLM READY


In [ ]:
!tail -n 100 /content/vllm.log

(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347] 
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347] 
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'host': '127.0.0.1', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85}
(APIServer pid=9154) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=9154) INFO 09-11 10:53:19 [model.py:684] Resolved 

In [6]:
import requests
import time

URL = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain AI inference in one sentence."
        }
    ],
    "max_tokens": 32,
    "temperature": 0
}

start = time.perf_counter()

r = requests.post(
    URL,
    json=payload,
    timeout=60
)

elapsed = time.perf_counter() - start

print("status:", r.status_code)
print("latency:", round(elapsed, 3), "seconds")

data = r.json()

print("response:")
print(data["choices"][0]["message"]["content"])

print("usage:")
print(data["usage"])

status: 200
latency: 0.924 seconds
response:
AI inference involves processing and analyzing large amounts of data to make predictions or decisions based on patterns and relationships within that data.
usage:
{'prompt_tokens': 37, 'total_tokens': 62, 'completion_tokens': 25, 'prompt_tokens_details': None, 'completion_tokens_details': None}


In [7]:
m = requests.get(
    "http://127.0.0.1:8000/metrics",
    timeout=10
)

print("metrics status:", m.status_code)

wanted = [
    "vllm:num_requests_running",
    "vllm:num_requests_waiting",
    "vllm:request_success_total",
    "vllm:e2e_request_latency_seconds",
    "vllm:request_queue_time_seconds",
    "vllm:prompt_tokens_total",
    "vllm:generation_tokens_total",
]

for metric in wanted:
    print("\n---", metric, "---")
    for line in m.text.splitlines():
        if line.startswith(metric):
            print(line)

metrics status: 200

--- vllm:num_requests_running ---
vllm:num_requests_running{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0

--- vllm:num_requests_waiting ---
vllm:num_requests_waiting{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="capacity"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="deferred"} 0.0

--- vllm:request_success_total ---
vllm:request_success_total{engine="0",finished_reason="stop",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 1.0
vllm:request_success_total{engine="0",finished_reason="length",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="abort",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="error",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="repetit

# background metrics sampler

In [8]:
import threading
import requests
import time
import re
import csv

stop_sampling = False
samples = []

def get_value(text, name):
    pattern = rf'^{re.escape(name)}\{{.*?\}}\s+([0-9.eE+-]+)'
    m = re.search(pattern, text, re.MULTILINE)
    return float(m.group(1)) if m else None

def sampler():
    while not stop_sampling:
        try:
            text = requests.get(
                "http://127.0.0.1:8000/metrics",
                timeout=2
            ).text

            samples.append({
                "timestamp": time.time(),
                "running": get_value(
                    text,
                    "vllm:num_requests_running"
                ),
                "waiting": get_value(
                    text,
                    "vllm:num_requests_waiting"
                )
            })
        except Exception:
            pass

        time.sleep(0.25)

thread = threading.Thread(target=sampler, daemon=True)
thread.start()

print("sampler started")

sampler started


# concurrent overload test

In [9]:
import asyncio
import aiohttp
import time
import statistics

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30

async def send_one(session, i):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": EXPENSIVE_PROMPT}
        ],
        "max_tokens": 256,
        "temperature": 0
    }

    start = time.perf_counter()

    try:
        async with session.post(URL, json=payload, timeout=120) as r:
            await r.text()
            latency = time.perf_counter() - start
            return {
                "id": i,
                "status": r.status,
                "latency": latency
            }
    except Exception as e:
        return {
            "id": i,
            "status": "error",
            "latency": time.perf_counter() - start
        }



async def run_load(n=50):
    async with aiohttp.ClientSession() as session:
        tasks = [
            asyncio.create_task(send_one(session, i))
            for i in range(n)
        ]
        return await asyncio.gather(*tasks)

results = await run_load(40)

latencies = [
    r["latency"]
    for r in results
    if r["status"] == 200
]

print("completed:", len(results))
print("successful:", len(latencies))

if latencies:
    print("p50:", round(statistics.median(latencies), 3))

    sorted_lat = sorted(latencies)
    p95_index = int(0.95 * len(sorted_lat)) - 1
    print("p95:", round(sorted_lat[p95_index], 3))

    print("max:", round(max(latencies), 3))

completed: 40
successful: 40
p50: 7.209
p95: 11.833
max: 11.834


In [10]:
stop_sampling = True
thread.join(timeout=2)

print("samples:", len(samples))
print("max running:", max(x["running"] or 0 for x in samples))
print("max waiting:", max(x["waiting"] or 0 for x in samples))

samples: 46
max running: 8.0
max waiting: 32.0


# Load generator

In [11]:
!pip install -q locust
!locust --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 22.2 MB/s eta 0:00:00
locust 2.46.5 from /usr/local/lib/python3.13/dist-packages/locust (Python 3.13.15)


In [12]:
%%writefile /content/locustfile.py
# first testing with normal traffic
from locust import HttpUser, task, between
import random


MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 4

    @task
    def legitimate_request(self):
        payload = {
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": SMALL_PROMPT
                }
            ],
            "max_tokens": 32,
            "temperature": 0
        }

        self.client.post(
            "/v1/chat/completions",
            json=payload,
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(0.05, 0.15)
    weight = 1

    @task
    def attack_request(self):
        payload = {
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": EXPENSIVE_PROMPT
                }
            ],
            "max_tokens": 256,
            "temperature": 0
        }

        self.client.post(
            "/v1/chat/completions",
            json=payload,
            name="attacker"
        )

Writing /content/locustfile.py


In [13]:
!mkdir -p /content/results

In [14]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --users 4 \
  --spawn-rate 2 \
  --run-time 15s \
  --csv /content/results/normal \
  --csv-full-history \
  LegitimateUser

[2026-09-11 14:07:45,665] 626a8ef1d973/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 14:07:45,666] 626a8ef1d973/INFO/locust.main: Run time limit set to 15 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 14:07:45,668] 626a8ef1d973/INFO/locust.runners: Ramping to 4 users at a rate of 2.00 per second
[2026-09-11 14:07:46,669] 626a8ef1d973/INFO/locust.runners: All users spawned: {"LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     legitimate       7     0(0.00%) |    226     197     286    210 |  

In [15]:
import pandas as pd

df = pd.read_csv("/content/results/normal_stats.csv")
display(df)

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,legitimate,47,0,210,222.495028,197.791084,317.907559,870.0,3.356832,...,210,220,250,290,300,320,320,320,320,320
1,NaN,Aggregated,47,0,210,222.495028,197.791084,317.907559,870.0,3.356832,...,210,220,250,290,300,320,320,320,320,320


In [16]:
%%writefile /content/locustfile.py
# sudden spike
from locust import HttpUser, task, between, LoadTestShape

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 1

    @task
    def legitimate_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": SMALL_PROMPT
                    }
                ],
                "max_tokens": 32,
                "temperature": 0
            },
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(0.05, 0.15)
    weight = 1

    @task
    def attack_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": MEDIUM_PROMPT
                    }
                ],
                "max_tokens": 128,
                "temperature": 0
            },
            name="attacker"
        )


class SpikeShape(LoadTestShape):

    def tick(self):
        run_time = self.get_run_time()

        # 0-10s: legitimate traffic only
        if run_time < 10:
            return (
                4,
                4,
                [LegitimateUser]
            )

        # 10-20s: sudden attacker spike
        if run_time < 20:
            return (
                24,
                20,
                [LegitimateUser, AttackerUser]
            )

        # 20-30s: attack stops, return to normal
        if run_time < 30:
            return (
                4,
                20,
                [LegitimateUser]
            )

        return None

Overwriting /content/locustfile.py


In [17]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --csv /content/results/spike \
  --csv-full-history

[2026-09-11 14:08:02,913] 626a8ef1d973/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 14:08:02,914] 626a8ef1d973/INFO/locust.runners: Shape test starting.
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 14:08:02,915] 626a8ef1d973/INFO/locust.runners: Shape worker starting
[2026-09-11 14:08:02,915] 626a8ef1d973/INFO/locust.runners: Shape test updating to 4 users at 4.00 spawn rate
[2026-09-11 14:08:02,916] 626a8ef1d973/INFO/locust.runners: Ramping to 4 users at a rate of 4.00 per second
[2026-09-11 14:08:02,916] 626a8ef1d973/INFO/locust.runners: All users spawned: {"AttackerUser": 0, "LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     

In [18]:
import pandas as pd

df = pd.read_csv("/content/results/spike_stats.csv")
display(df)

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,attacker,63,0,1400,1417.387815,973.886594,1911.976599,1414.0,2.100442,...,1500,1600,1600,1700,1700,1900,1900,1900,1900,1900
1,POST,legitimate,147,0,480,535.533176,196.823806,1450.673655,870.0,4.901032,...,720,800,850,960,1200,1400,1400,1500,1500,1500
2,NaN,Aggregated,210,0,770,800.089567,196.823806,1911.976599,1033.2,7.001475,...,1000,1300,1400,1500,1700,1700,1800,1900,1900,1900


In [19]:
hist = pd.read_csv("/content/results/spike_stats_history.csv")

display(hist.tail(20))

,Timestamp,User Count,Type,Name,Requests/s,Failures/s,50%,66%,75%,80%,...,99.9%,99.99%,100%,Total Request Count,Total Failure Count,Total Median Response Time,Total Average Response Time,Total Min Response Time,Total Max Response Time,Total Average Content Size
61,1789135706,4,POST,legitimate,8.0,0.0,730.0,790.0,840.0,850.0,...,1000.0,1000.0,1000.0,127,0,620.0,585.937106,196.823806,1450.673655,870.000000
62,1789135706,4,NaN,Aggregated,13.6,0.0,860.0,1300.0,1400.0,1500.0,...,1700.0,1700.0,1700.0,190,0,840.0,861.628656,196.823806,1911.976599,1050.378947
63,1789135707,4,POST,attacker,5.3,0.0,1500.0,1500.0,1600.0,1600.0,...,1700.0,1700.0,1700.0,63,0,1400.0,1417.387815,973.886594,1911.976599,1414.000000
64,1789135707,4,POST,legitimate,7.1,0.0,730.0,820.0,850.0,860.0,...,1000.0,1000.0,1000.0,130,0,590.0,577.104274,196.823806,1450.673655,870.000000
65,1789135707,4,NaN,Aggregated,12.4,0.0,860.0,1200.0,1400.0,1500.0,...,1700.0,1700.0,1700.0,193,0,830.0,851.393720,196.823806,1911.976599,1047.575130
66,1789135708,4,POST,attacker,4.5,0.0,1500.0,1600.0,1600.0,1700.0,...,1700.0,1700.0,1700.0,63,0,1400.0,1417.387815,973.886594,1911.976599,1414.000000
67,1789135708,4,POST,legitimate,6.6,0.0,680.0,770.0,840.0,860.0,...,1000.0,1000.0,1000.0,133,0,590.0,568.906093,196.823806,1450.673655,870.000000
68,1789135708,4,NaN,Aggregated,11.1,0.0,850.0,1200.0,1400.0,1500.0,...,1700.0,1700.0,1700.0,196,0,820.0,841.632361,196.823806,1911.976599,1044.857143
69,1789135709,4,POST,attacker,3.8,0.0,1600.0,1600.0,1700.0,1700.0,...,1700.0,1700.0,1700.0,63,0,1400.0,1417.387815,973.886594,1911.976599,1414.000000
70,1789135709,4,POST,legitimate,6.2,0.0,590.0,720.0,820.0,870.0,...,1000.0,1000.0,1000.0,135,0,560.0,563.543834,196.823806,1450.673655,870.000000


In [20]:
%%writefile /content/locustfile.py
# sustained flood
from locust import HttpUser, task, between, LoadTestShape

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 1

    @task
    def legitimate_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": SMALL_PROMPT
                    }
                ],
                "max_tokens": 32,
                "temperature": 0
            },
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(0.05, 0.15)
    weight = 1

    @task
    def attack_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": MEDIUM_PROMPT
                    }
                ],
                "max_tokens": 128,
                "temperature": 0
            },
            name="attacker"
        )


class FloodShape(LoadTestShape):

    def tick(self):
        run_time = self.get_run_time()

        # 0-10s: normal legitimate traffic
        if run_time < 10:
            return (
                4,
                4,
                [LegitimateUser]
            )

        # 10-40s: sustained flood
        if run_time < 40:
            return (
                32,
                20,
                [LegitimateUser, AttackerUser]
            )

        # 40-50s: recovery
        if run_time < 50:
            return (
                4,
                20,
                [LegitimateUser]
            )

        return None

Overwriting /content/locustfile.py


In [21]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --csv /content/results/flood \
  --csv-full-history

[2026-09-11 14:08:35,177] 626a8ef1d973/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 14:08:35,178] 626a8ef1d973/INFO/locust.runners: Shape test starting.
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 14:08:35,179] 626a8ef1d973/INFO/locust.runners: Shape worker starting
[2026-09-11 14:08:35,179] 626a8ef1d973/INFO/locust.runners: Shape test updating to 4 users at 4.00 spawn rate
[2026-09-11 14:08:35,179] 626a8ef1d973/INFO/locust.runners: Ramping to 4 users at a rate of 4.00 per second
[2026-09-11 14:08:35,180] 626a8ef1d973/INFO/locust.runners: All users spawned: {"AttackerUser": 0, "LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     

In [22]:
import pandas as pd

df = pd.read_csv("/content/results/flood_stats.csv")
display(df)

hist = pd.read_csv("/content/results/flood_stats_history.csv")
display(hist.tail(30))

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,attacker,187,0,2100,2096.167027,894.661942,2693.519002,1413.860963,3.753275,...,2200,2300,2300,2400,2500,2600,2600,2700,2700,2700
1,POST,legitimate,292,0,1400,1158.508394,200.183667,1948.905035,870.000000,5.860728,...,1500,1600,1600,1700,1800,1800,1900,1900,1900,1900
2,NaN,Aggregated,479,0,1600,1524.567192,200.183667,2693.519002,1082.321503,9.614003,...,1900,2100,2100,2300,2400,2500,2600,2700,2700,2700


,Timestamp,User Count,Type,Name,Requests/s,Failures/s,50%,66%,75%,80%,...,99.9%,99.99%,100%,Total Request Count,Total Failure Count,Total Median Response Time,Total Average Response Time,Total Min Response Time,Total Max Response Time,Total Average Content Size
110,1789135756,12,POST,attacker,6.1,0.0,2200.0,2300.0,2300.0,2400.0,...,2600.0,2600.0,2600.0,187,0,2100,2096.167027,894.661942,2693.519002,1413.860963
111,1789135756,12,POST,legitimate,7.7,0.0,1500.0,1600.0,1600.0,1600.0,...,1800.0,1800.0,1800.0,263,0,1400,1261.670528,200.183667,1948.905035,870.000000
112,1789135756,12,NaN,Aggregated,13.8,0.0,1600.0,2100.0,2200.0,2300.0,...,2600.0,2600.0,2600.0,450,0,1600,1608.450184,200.183667,2693.519002,1096.004444
113,1789135757,4,POST,attacker,6.1,0.0,2300.0,2300.0,2400.0,2400.0,...,2600.0,2600.0,2600.0,187,0,2100,2096.167027,894.661942,2693.519002,1413.860963
114,1789135757,4,POST,legitimate,7.7,0.0,1500.0,1600.0,1600.0,1600.0,...,1800.0,1800.0,1800.0,266,0,1400,1249.936148,200.183667,1948.905035,870.000000
115,1789135757,4,NaN,Aggregated,13.8,0.0,1600.0,2100.0,2200.0,2300.0,...,2600.0,2600.0,2600.0,453,0,1600,1599.263244,200.183667,2693.519002,1094.507726
116,1789135758,4,POST,attacker,5.8,0.0,2300.0,2300.0,2400.0,2400.0,...,2600.0,2600.0,2600.0,187,0,2100,2096.167027,894.661942,2693.519002,1413.860963
117,1789135758,4,POST,legitimate,7.4,0.0,1500.0,1600.0,1600.0,1600.0,...,1800.0,1800.0,1800.0,269,0,1400,1238.455772,200.183667,1948.905035,870.000000
118,1789135758,4,NaN,Aggregated,13.2,0.0,1600.0,2100.0,2200.0,2300.0,...,2600.0,2600.0,2600.0,456,0,1600,1590.192624,200.183667,2693.519002,1093.030702
119,1789135759,4,POST,attacker,5.5,0.0,2300.0,2400.0,2400.0,2400.0,...,2600.0,2600.0,2600.0,187,0,2100,2096.167027,894.661942,2693.519002,1413.860963


In [23]:
%%writefile /content/locustfile.py
# low-and-slow
from locust import HttpUser, task, between, LoadTestShape

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 1

    @task
    def legitimate_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": SMALL_PROMPT
                    }
                ],
                "max_tokens": 32,
                "temperature": 0
            },
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(1.5, 2.5)

    @task
    def attack_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": EXPENSIVE_PROMPT
                    }
                ],
                "max_tokens": 256,
                "temperature": 0
            },
            name="attacker"
        )


class LowSlowShape(LoadTestShape):

    def tick(self):
        run_time = self.get_run_time()

        # 0-10s: normal traffic only
        if run_time < 10:
            return (
                4,
                4,
                [LegitimateUser]
            )

        # 10-40s: low-rate expensive attackers appear
        if run_time < 40:
            return (
                8,
                2,
                [LegitimateUser, AttackerUser]
            )

        # 40-50s: attacker disappears
        if run_time < 50:
            return (
                4,
                4,
                [LegitimateUser]
            )

        return None

Overwriting /content/locustfile.py


In [24]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --csv /content/results/low_slow \
  --csv-full-history

[2026-09-11 14:09:27,091] 626a8ef1d973/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 14:09:27,092] 626a8ef1d973/INFO/locust.runners: Shape test starting.
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 14:09:27,093] 626a8ef1d973/INFO/locust.runners: Shape worker starting
[2026-09-11 14:09:27,093] 626a8ef1d973/INFO/locust.runners: Shape test updating to 4 users at 4.00 spawn rate
[2026-09-11 14:09:27,093] 626a8ef1d973/INFO/locust.runners: Ramping to 4 users at a rate of 4.00 per second
[2026-09-11 14:09:27,094] 626a8ef1d973/INFO/locust.runners: All users spawned: {"AttackerUser": 0, "LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     

In [25]:
import pandas as pd

df = pd.read_csv("/content/results/low_slow_stats.csv")
display(df)

hist = pd.read_csv("/content/results/low_slow_stats_history.csv")
display(hist.tail(30))

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,attacker,16,0,1891.112628,1859.483856,1820.355737,1891.112628,2114.000000,0.322025,...,1900,1900,1900,1900,1900,1900,1900,1900,1900,1900
1,POST,legitimate,213,0,240.000000,235.768356,204.112324,280.653782,870.000000,4.286959,...,240,250,250,260,260,260,270,280,280,280
2,NaN,Aggregated,229,0,240.000000,349.215727,204.112324,1891.112628,956.917031,4.608984,...,250,250,260,260,1900,1900,1900,1900,1900,1900


,Timestamp,User Count,Type,Name,Requests/s,Failures/s,50%,66%,75%,80%,...,99.9%,99.99%,100%,Total Request Count,Total Failure Count,Total Median Response Time,Total Average Response Time,Total Min Response Time,Total Max Response Time,Total Average Content Size
110,1789135808,4,POST,attacker,0.5,0.0,1900.0,1900.0,1900.0,1900.0,...,1900.0,1900.0,1900.0,16,0,1891.112628,1859.483856,1820.355737,1891.112628,2114.000000
111,1789135808,4,POST,legitimate,5.0,0.0,240.0,250.0,250.0,260.0,...,280.0,280.0,280.0,185,0,240.000000,238.332297,204.295126,280.653782,870.000000
112,1789135808,4,NaN,Aggregated,5.5,0.0,240.0,250.0,260.0,260.0,...,1900.0,1900.0,1900.0,201,0,240.000000,367.379187,204.295126,1891.112628,969.024876
113,1789135809,4,POST,attacker,0.5,0.0,1900.0,1900.0,1900.0,1900.0,...,1900.0,1900.0,1900.0,16,0,1891.112628,1859.483856,1820.355737,1891.112628,2114.000000
114,1789135809,4,POST,legitimate,5.0,0.0,240.0,250.0,250.0,250.0,...,280.0,280.0,280.0,188,0,240.000000,237.919078,204.295126,280.653782,870.000000
115,1789135809,4,NaN,Aggregated,5.5,0.0,240.0,250.0,260.0,260.0,...,1900.0,1900.0,1900.0,204,0,240.000000,365.100629,204.295126,1891.112628,967.568627
116,1789135810,4,POST,attacker,0.5,0.0,1900.0,1900.0,1900.0,1900.0,...,1900.0,1900.0,1900.0,16,0,1891.112628,1859.483856,1820.355737,1891.112628,2114.000000
117,1789135810,4,POST,legitimate,4.8,0.0,240.0,240.0,250.0,250.0,...,280.0,280.0,280.0,191,0,240.000000,237.704557,204.295126,280.653782,870.000000
118,1789135810,4,NaN,Aggregated,5.3,0.0,240.0,250.0,250.0,260.0,...,1900.0,1900.0,1900.0,207,0,240.000000,363.059479,204.295126,1891.112628,966.154589
119,1789135811,4,POST,attacker,0.5,0.0,1900.0,1900.0,1900.0,1900.0,...,1900.0,1900.0,1900.0,16,0,1891.112628,1859.483856,1820.355737,1891.112628,2114.000000
